# Algoritmo de Borůvka

Es el algoritmo más antiguo para el árbol de expansión mínima (Otakar Borůvka, 1926, para la red eléctrica de Moravia). Trabaja por fases: cada componente elige su arista más barata hacia fuera y se añaden todas a la vez. Cada fase al menos divide entre dos el número de componentes, así que hay como mucho $\lceil \log_2 V \rceil$ fases de coste $O(E)$ y el total es $O(E \log V)$. Para que las aristas elegidas a la vez no formen ciclos, los empates de peso se rompen siempre con el mismo criterio. Como las componentes eligen independientemente, es la base de los algoritmos paralelos y distribuidos de MST.

## 1. Cable submarino entre islas

Unir las islas Canarias con cable submarino de longitud total mínima. Implementar Borůvka con Union-Find: en cada fase, recorrer las aristas guardando para cada componente la más barata que sale de ella (comparando por `(peso, índice)`) y unir las componentes con las elegidas. Mostrar las aristas que se añaden en cada fase.

In [1]:
import math
import random
from itertools import combinations

import networkx as nx
import numpy as np
from scipy.sparse import csr_array
from scipy.sparse.csgraph import minimum_spanning_tree


class UnionFind:
    def __init__(self, nodos):
        self.padre = {n: n for n in nodos}
        self.componentes = len(self.padre)

    def find(self, x):
        while self.padre[x] != x:
            self.padre[x] = self.padre[self.padre[x]]
            x = self.padre[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        self.padre[rb] = ra
        self.componentes -= 1
        return True


def boruvka(nodos, aristas, traza=False):
    """aristas: lista de (u, v, peso). Devuelve (árbol, número de fases)."""
    uf = UnionFind(nodos)
    arbol, fases = [], 0
    while True:
        mejor = {}
        for i, (u, v, w) in enumerate(aristas):
            ru, rv = uf.find(u), uf.find(v)
            if ru != rv:
                for r in (ru, rv):
                    if r not in mejor or (w, i) < mejor[r]:
                        mejor[r] = (w, i)
        if not mejor:
            return arbol, fases
        fases += 1
        nuevas = []
        for _, i in sorted(set(mejor.values())):
            u, v, w = aristas[i]
            if uf.union(u, v):
                nuevas.append(aristas[i])
        arbol += nuevas
        if traza:
            print(f"Fase {fases}: +{len(nuevas)} cables, componentes que quedan: {uf.componentes}")
            for u, v, w in nuevas:
                print(f"    {u} – {v} ({w} km)")


islas = ["Lanzarote", "Fuerteventura", "Gran Canaria", "Tenerife", "La Gomera", "La Palma", "El Hierro", "La Graciosa"]
cables = [
    ("La Graciosa", "Lanzarote", 2), ("Lanzarote", "Fuerteventura", 11), ("Fuerteventura", "Gran Canaria", 85),
    ("Lanzarote", "Gran Canaria", 175), ("Gran Canaria", "Tenerife", 62), ("Tenerife", "La Gomera", 28),
    ("Tenerife", "La Palma", 85), ("La Gomera", "La Palma", 60), ("La Gomera", "El Hierro", 64),
    ("Tenerife", "El Hierro", 110), ("La Palma", "El Hierro", 75), ("Fuerteventura", "Tenerife", 160),
]
arbol, fases = boruvka(islas, cables, traza=True)
print("Longitud total:", sum(w for *_, w in arbol), "km en", len(arbol), "cables")
assert len(arbol) == len(islas) - 1

Fase 1: +6 cables, componentes que quedan: 2
    La Graciosa – Lanzarote (2 km)
    Lanzarote – Fuerteventura (11 km)
    Tenerife – La Gomera (28 km)
    La Gomera – La Palma (60 km)
    Gran Canaria – Tenerife (62 km)
    La Gomera – El Hierro (64 km)
Fase 2: +1 cables, componentes que quedan: 1
    Fuerteventura – Gran Canaria (85 km)
Longitud total: 312 km en 7 cables


## 2. Comprobación contra Kruskal, Prim y networkx

Escribir versiones cortas de Kruskal (ordenar y unir) y Prim (con `heapq`) y comprobar en muchos grafos aleatorios conexos que los tres dan el mismo peso total, y también `nx.minimum_spanning_tree` con `algorithm="boruvka"`.

In [2]:
import heapq


def kruskal(nodos, aristas):
    uf = UnionFind(nodos)
    return [(u, v, w) for u, v, w in sorted(aristas, key=lambda a: a[2]) if uf.union(u, v)]


def prim(nodos, aristas):
    vecinos = {n: [] for n in nodos}
    for u, v, w in aristas:
        vecinos[u].append((w, u, v))
        vecinos[v].append((w, v, u))
    inicio = next(iter(nodos))
    visitados, arbol = {inicio}, []
    cola = list(vecinos[inicio])
    heapq.heapify(cola)
    while cola:
        w, u, v = heapq.heappop(cola)
        if v not in visitados:
            visitados.add(v)
            arbol.append((u, v, w))
            for arista in vecinos[v]:
                heapq.heappush(cola, arista)
    return arbol


def grafo_conexo(n, p, rnd, pesos=100):
    """Árbol aleatorio más aristas extra con probabilidad p."""
    aristas = {(rnd.randrange(v), v) for v in range(1, n)}
    aristas |= {(u, v) for u, v in combinations(range(n), 2) if rnd.random() < p}
    return list(range(n)), [(u, v, rnd.randint(1, pesos)) for u, v in sorted(aristas)]


def peso(arbol):
    return sum(w for *_, w in arbol)


rnd = random.Random(12)
for _ in range(200):
    nodos, aristas = grafo_conexo(rnd.randint(1, 40), rnd.random() * 0.3, rnd)
    arbol, _ = boruvka(nodos, aristas)
    g = nx.Graph()
    g.add_nodes_from(nodos)
    g.add_weighted_edges_from(aristas)
    referencia = nx.minimum_spanning_tree(g, algorithm="boruvka").size(weight="weight")
    assert len(arbol) == len(nodos) - 1
    assert peso(arbol) == peso(kruskal(nodos, aristas)) == peso(prim(nodos, aristas)) == referencia
print("Borůvka, Kruskal, Prim y networkx coinciden en 200 grafos")

Borůvka, Kruskal, Prim y networkx coinciden en 200 grafos


## 3. Como mucho log₂ V fases

En cada fase toda componente se une al menos con otra, así que el número de componentes se divide al menos entre dos. Registrar cuántas componentes quedan tras cada fase en grafos grandes de distintas formas (aleatorio disperso, rejilla, camino) y comprobar que las fases no pasan de $\lceil \log_2 V \rceil$.

In [3]:
def componentes_por_fase(nodos, aristas):
    uf = UnionFind(nodos)
    historia = [uf.componentes]
    while uf.componentes > 1:
        mejor = {}
        for i, (u, v, w) in enumerate(aristas):
            ru, rv = uf.find(u), uf.find(v)
            if ru != rv:
                for r in (ru, rv):
                    mejor[r] = min(mejor.get(r, (w, i)), (w, i))
        if not mejor:
            break
        for _, i in mejor.values():
            uf.union(*aristas[i][:2])
        historia.append(uf.componentes)
    return historia


lado = 40
rejilla = [((f, c), (f + df, c + dc), rnd.randint(1, 1000)) for f in range(lado) for c in range(lado)
           for df, dc in ((0, 1), (1, 0)) if f + df < lado and c + dc < lado]
casos = {
    "aleatorio": grafo_conexo(2000, 0.002, rnd, pesos=10**6),
    "rejilla": ([(f, c) for f in range(lado) for c in range(lado)], rejilla),
    "camino": (list(range(2048)), [(v, v + 1, rnd.randint(1, 10**6)) for v in range(2047)]),
}
for nombre, (nodos, aristas) in casos.items():
    historia = componentes_por_fase(nodos, aristas)
    cota = math.ceil(math.log2(len(nodos)))
    print(f"{nombre:10} V = {len(nodos):>4}: {len(historia) - 1} fases (cota {cota}) → {historia}")
    assert all(despues <= antes // 2 for antes, despues in zip(historia, historia[1:]))
    assert len(historia) - 1 <= cota and historia[-1] == 1

aleatorio  V = 2000: 5 fases (cota 11) → [2000, 513, 106, 20, 6, 1]
rejilla    V = 1600: 6 fases (cota 11) → [1600, 452, 121, 32, 8, 2, 1]
camino     V = 2048: 7 fases (cota 11) → [2048, 678, 218, 70, 26, 9, 3, 1]


## 4. Aristas de igual peso

Si se elige "la arista más barata" sin un criterio de desempate común, con pesos iguales cada componente puede elegir una arista distinta y juntas formar un ciclo. Mostrarlo con un triángulo de pesos iguales, añadiendo a ciegas las aristas elegidas (sin comprobar con Union-Find), y comprobar que con el desempate `(peso, índice)` las aristas elegidas en cada fase nunca forman ciclo en grafos aleatorios con muchos empates.

In [4]:
def elegidas_primera_fase(nodos, aristas, clave):
    elegida = {}
    for i, (u, v, w) in enumerate(aristas):
        for r in (u, v):
            if r not in elegida or clave(i, r) < clave(elegida[r], r):
                elegida[r] = i
    return {aristas[i] for i in elegida.values()}


triangulo = [("A", "B", 5), ("B", "C", 5), ("C", "A", 5)]
# en caso de empate, cada nodo prefiere la arista en la que aparece primero: el criterio depende del nodo
sin_criterio = elegidas_primera_fase("ABC", triangulo, clave=lambda i, r: (triangulo[i][2], triangulo[i][0] != r))
con_criterio = elegidas_primera_fase("ABC", triangulo, clave=lambda i, r: (triangulo[i][2], i))
print("Sin desempate común:", sorted(sin_criterio), "→ ciclo de", len(sin_criterio), "aristas")
print("Con (peso, índice):  ", sorted(con_criterio))
assert len(sin_criterio) == 3 and len(con_criterio) == 2


def sin_ciclos(nodos, aristas):
    uf = UnionFind(nodos)
    return all(uf.union(u, v) for u, v, _ in aristas)


for _ in range(300):
    nodos, aristas = grafo_conexo(rnd.randint(2, 30), 0.3, rnd, pesos=3)
    assert sin_ciclos(nodos, elegidas_primera_fase(nodos, aristas, clave=lambda i, r: (aristas[i][2], i)))

Sin desempate común: [('A', 'B', 5), ('B', 'C', 5), ('C', 'A', 5)] → ciclo de 3 aristas
Con (peso, índice):   [('A', 'B', 5), ('B', 'C', 5)]


## 5. ¿Un árbol o varios?

Con pesos distintos el MST es único, así que Borůvka, Kruskal y Prim devuelven exactamente las mismas aristas. Con empates puede haber varios MST del mismo peso. Contar los MST de grafos pequeños probando todos los subconjuntos de $V - 1$ aristas, y comprobar que con pesos distintos sale siempre 1 y que con empates todos coinciden en peso, pero el árbol concreto depende del desempate: Borůvka y Kruskal con el mismo criterio (peso, posición) eligen el mismo, y Kruskal con las aristas en orden inverso puede elegir otro.

In [5]:
def contar_mst(nodos, aristas):
    minimo, cuenta = math.inf, 0
    for subconjunto in combinations(aristas, len(nodos) - 1):
        if sin_ciclos(nodos, subconjunto):
            p = peso(subconjunto)
            if p < minimo:
                minimo, cuenta = p, 1
            elif p == minimo:
                cuenta += 1
    return cuenta, minimo


def normaliza(arbol):
    return sorted((min(u, v), max(u, v), w) for u, v, w in arbol)


varios, distintos = 0, 0
for _ in range(150):
    nodos, aristas = grafo_conexo(rnd.randint(2, 6), 0.5, rnd)
    pesos = rnd.sample(range(1, 100), len(aristas))
    unicos = [(u, v, w) for (u, v, _), w in zip(aristas, pesos)]
    assert contar_mst(nodos, unicos)[0] == 1
    assert normaliza(boruvka(nodos, unicos)[0]) == normaliza(kruskal(nodos, unicos)) == normaliza(prim(nodos, unicos))

    empatados = [(u, v, rnd.randint(1, 2)) for u, v, _ in aristas]
    cuenta, minimo = contar_mst(nodos, empatados)
    b, k, k_inverso = boruvka(nodos, empatados)[0], kruskal(nodos, empatados), kruskal(nodos, empatados[::-1])
    assert all(peso(a) == minimo for a in (b, k, k_inverso, prim(nodos, empatados)))
    # mismo desempate (peso, posición) → mismo árbol
    assert normaliza(b) == normaliza(k)
    varios += cuenta > 1
    distintos += normaliza(b) != normaliza(k_inverso)

cuadrado = [(0, 1, 1), (1, 2, 1), (2, 3, 1), (3, 0, 1)]
print("Ciclo de 4 aristas iguales:", contar_mst(range(4), cuadrado)[0], "árboles mínimos distintos")
print(f"Grafos con empates y varios MST: {varios} de 150; en {distintos} cambia el árbol al invertir el orden de las aristas")

Ciclo de 4 aristas iguales: 4 árboles mínimos distintos
Grafos con empates y varios MST: 76 de 150; en 76 cambia el árbol al invertir el orden de las aristas


## 6. Bosque de expansión mínima

Si el grafo no es conexo, Borůvka termina cuando ninguna componente tiene aristas hacia fuera y devuelve un bosque con $V - C$ aristas, siendo $C$ el número de componentes conexas. Probarlo con las redes eléctricas de islas sin conexión entre sí y comprobar $C$ con `nx.number_connected_components`.

In [6]:
redes = [
    ("Arrecife", "Tías", 9), ("Tías", "Yaiza", 12), ("Arrecife", "Teguise", 11), ("Teguise", "Haría", 14), ("Tías", "Teguise", 15),
    ("Puerto del Rosario", "Antigua", 19), ("Antigua", "Tuineje", 18), ("Puerto del Rosario", "Tuineje", 30),
    ("Valverde", "Frontera", 20),
]
pueblos = sorted({p for u, v, _ in redes for p in (u, v)})
bosque, fases = boruvka(pueblos, redes)
g = nx.Graph((u, v) for u, v, _ in redes)
c = nx.number_connected_components(g)
print(f"{len(pueblos)} pueblos, {c} redes, {len(bosque)} líneas, {peso(bosque)} km, fases: {fases}")
for u, v, w in bosque:
    print(f"    {u} – {v} ({w} km)")
assert len(bosque) == len(pueblos) - c

for _ in range(200):
    n = rnd.randint(1, 25)
    aristas = [(u, v, rnd.randint(1, 50)) for u, v in combinations(range(n), 2) if rnd.random() < 0.1]
    g = nx.Graph()
    g.add_nodes_from(range(n))
    g.add_weighted_edges_from(aristas)
    bosque, _ = boruvka(range(n), aristas)
    assert len(bosque) == n - nx.number_connected_components(g)
    assert peso(bosque) == nx.minimum_spanning_tree(g).size(weight="weight")

10 pueblos, 3 redes, 7 líneas, 103 km, fases: 1
    Arrecife – Tías (9 km)
    Arrecife – Teguise (11 km)
    Tías – Yaiza (12 km)
    Teguise – Haría (14 km)
    Antigua – Tuineje (18 km)
    Puerto del Rosario – Antigua (19 km)
    Valverde – Frontera (20 km)


## 7. Contracción explícita de componentes

Otra forma de verlo: tras cada fase, contraer cada componente en un solo nodo, borrar las aristas que quedan dentro de una componente y, entre cada par de componentes, quedarse solo con la arista más barata. Así el grafo encoge de verdad y cada fase trabaja con menos aristas. Implementarlo, mostrar nodos y aristas por fase en un grafo aleatorio y comprobar que el peso total coincide con la versión con Union-Find.

In [7]:
def boruvka_contrayendo(n, aristas):
    """Nodos 0..n-1; aristas (u, v, peso, id). Devuelve las id del árbol y el tamaño del grafo en cada fase."""
    elegidas, tamanos = [], []
    while aristas:
        tamanos.append((n, len(aristas)))
        mejor = [None] * n
        for a in aristas:
            for extremo in a[:2]:
                if mejor[extremo] is None or a[2:] < mejor[extremo][2:]:
                    mejor[extremo] = a
        uf = UnionFind(range(n))
        for a in set(m for m in mejor if m is not None):
            uf.union(a[0], a[1])
            elegidas.append(a[3])
        raices = sorted({uf.find(v) for v in range(n)})
        nuevo = {r: k for k, r in enumerate(raices)}
        mas_barata = {}
        for u, v, w, i in aristas:
            cu, cv = nuevo[uf.find(u)], nuevo[uf.find(v)]
            if cu != cv:
                clave = (min(cu, cv), max(cu, cv))
                if clave not in mas_barata or (w, i) < mas_barata[clave][2:]:
                    mas_barata[clave] = (cu, cv, w, i)
        n, aristas = len(raices), list(mas_barata.values())
    tamanos.append((n, 0))
    return elegidas, tamanos


nodos, aristas = grafo_conexo(3000, 0.001, rnd, pesos=10**6)
ids, tamanos = boruvka_contrayendo(len(nodos), [(u, v, w, i) for i, (u, v, w) in enumerate(aristas)])
print(f"{'fase':>4} {'nodos':>6} {'aristas':>8}")
for fase, (n, m) in enumerate(tamanos):
    print(f"{fase:>4} {n:>6} {m:>8}")
assert len(ids) == len(nodos) - 1 and peso(aristas[i] for i in ids) == peso(boruvka(nodos, aristas)[0])

for _ in range(100):
    nodos, aristas = grafo_conexo(rnd.randint(1, 30), 0.3, rnd, pesos=5)
    ids, _ = boruvka_contrayendo(len(nodos), [(u, v, w, i) for i, (u, v, w) in enumerate(aristas)])
    assert sorted(ids) == sorted(aristas.index(a) for a in boruvka(nodos, aristas)[0])

fase  nodos  aristas
   0   3000     7468
   1    770     5178
   2    169     3405
   3     31      430
   4      4        6
   5      1        0


## 8. Estaciones meteorológicas en el plano

Conectar 300 estaciones repartidas al azar con un grafo completo de distancias euclídeas. Calcular el MST con Borůvka y compararlo con `scipy.sparse.csgraph.minimum_spanning_tree` sobre la matriz de distancias. Comprobar también la propiedad de que la arista más corta que sale de cada punto (la de su vecino más cercano) siempre está en el MST, que es justo lo que Borůvka añade en la primera fase.

In [8]:
rng = np.random.default_rng(21)
puntos = rng.random((300, 2)) * 100
distancias = np.linalg.norm(puntos[:, None] - puntos[None, :], axis=2)
completo = [(i, j, float(distancias[i, j])) for i, j in combinations(range(len(puntos)), 2)]

arbol, fases = boruvka(range(len(puntos)), completo)
referencia = minimum_spanning_tree(csr_array(distancias)).sum()
print(f"Borůvka: {peso(arbol):.3f} en {fases} fases; SciPy: {referencia:.3f}")
assert math.isclose(peso(arbol), referencia)

en_arbol = {(min(u, v), max(u, v)) for u, v, _ in arbol}
np.fill_diagonal(distancias, np.inf)
vecino = distancias.argmin(axis=1)
assert all((min(i, j), max(i, j)) in en_arbol for i, j in enumerate(vecino))
print("Todas las aristas al vecino más cercano están en el MST:", len({(min(i, j), max(i, j)) for i, j in enumerate(vecino)}), "aristas distintas")

Borůvka: 1145.146 en 5 fases; SciPy: 1145.146
Todas las aristas al vecino más cercano están en el MST: 207 aristas distintas


## 9. Árbol de cuello de botella mínimo

En una red de senderos con desniveles, un excursionista quiere unir dos refugios de forma que la mayor subida de un tramo sea la menor posible. El MST es también un árbol de cuello de botella mínimo: el camino entre dos nodos dentro del MST minimiza la arista más pesada. Comprobarlo contra la fuerza bruta "menor umbral $t$ con el que $u$ y $v$ quedan conectados usando solo aristas de peso $\le t$".

In [9]:
def cuello_en_arbol(arbol, u, v):
    g = nx.Graph()
    g.add_weighted_edges_from(arbol)
    camino = nx.shortest_path(g, u, v)
    return max((g.edges[a, b]["weight"] for a, b in zip(camino, camino[1:])), default=0)


def cuello_bruto(nodos, aristas, u, v):
    for t in sorted({0} | {w for *_, w in aristas}):
        uf = UnionFind(nodos)
        for a, b, w in aristas:
            if w <= t:
                uf.union(a, b)
        if uf.find(u) == uf.find(v):
            return t


senderos = [
    ("Cruz del Carmen", "Chinamada", 320), ("Chinamada", "Punta del Hidalgo", 610), ("Cruz del Carmen", "Taborno", 280),
    ("Taborno", "Afur", 350), ("Afur", "Taganana", 400), ("Chinamada", "Afur", 150), ("Cruz del Carmen", "Chamorga", 700),
    ("Taganana", "Chamorga", 450), ("Punta del Hidalgo", "Taborno", 520),
]
refugios = sorted({p for u, v, _ in senderos for p in (u, v)})
arbol, _ = boruvka(refugios, senderos)
for u, v in [("Punta del Hidalgo", "Chamorga"), ("Taborno", "Taganana")]:
    print(f"{u} → {v}: mayor subida {cuello_en_arbol(arbol, u, v)} m")
    assert cuello_en_arbol(arbol, u, v) == cuello_bruto(refugios, senderos, u, v)

for _ in range(100):
    nodos, aristas = grafo_conexo(rnd.randint(2, 20), 0.3, rnd)
    arbol, _ = boruvka(nodos, aristas)
    for _ in range(10):
        u, v = rnd.sample(nodos, 2)
        assert cuello_en_arbol(arbol, u, v) == cuello_bruto(nodos, aristas, u, v)

Punta del Hidalgo → Chamorga: mayor subida 520 m
Taborno → Taganana: mayor subida 400 m


## 10. Árbol de expansión máxima

A veces interesa el árbol de peso máximo, por ejemplo para conectar repetidores usando los enlaces de mayor calidad de señal. Basta cambiar el signo de los pesos y aplicar Borůvka. Comprobarlo contra `nx.maximum_spanning_tree` y contra la fuerza bruta en grafos pequeños.

In [10]:
def arbol_maximo(nodos, aristas):
    arbol, _ = boruvka(nodos, [(u, v, -w) for u, v, w in aristas])
    return [(u, v, -w) for u, v, w in arbol]


senal = [("Izaña", "Teide", 92), ("Izaña", "Anaga", 75), ("Anaga", "Teno", 40), ("Teide", "Teno", 81),
         ("Teide", "Adeje", 88), ("Adeje", "Teno", 66), ("Anaga", "Adeje", 35)]
repetidores = sorted({p for u, v, _ in senal for p in (u, v)})
arbol = arbol_maximo(repetidores, senal)
print("Enlaces:", [(u, v) for u, v, _ in arbol], "calidad total", peso(arbol))

for _ in range(150):
    nodos, aristas = grafo_conexo(rnd.randint(2, 6), 0.5, rnd)
    g = nx.Graph()
    g.add_weighted_edges_from(aristas)
    g.add_nodes_from(nodos)
    arbol = arbol_maximo(nodos, aristas)
    maximo_bruto = max(peso(s) for s in combinations(aristas, len(nodos) - 1) if sin_ciclos(nodos, s))
    assert peso(arbol) == nx.maximum_spanning_tree(g).size(weight="weight") == maximo_bruto

Enlaces: [('Izaña', 'Teide'), ('Teide', 'Adeje'), ('Teide', 'Teno'), ('Izaña', 'Anaga')] calidad total 336
